<a id="top"></a>

# QFT repeated-secret study

**Question.** Can Eves pool sparse measurements across repeats of a four-bit secret and recover it before Bob alarms?

Run from the repository root. The default view loads the [saved 1,210-setting sweep](results/qft_factorial_results.csv). The first code cell below controls every optional run. All expensive run flags are off by default.

- [Configuration](#qft-settings)
- [Read the saved results](#qft-dashboard)
- [Follow one exchange and the stopping rules](#qft-methods)
- [Run a small validation and demo](#qft-demo)
- [Test channel drift and recalibration](#qft-recalibration)
- [Reproduce the full sweep](#qft-batch)
- [Trace the methods to their sources](#references)

See the [BB84 detector comparison](BB84.ipynb) for pooled QBER, U2, and U4 and the [project README](README.md) for the overall study.

<a id="qft-settings"></a>

## Configuration

Edit this one settings cell, then run the notebook from the top. Paths are relative to the repository root.

In [ ]:
from pathlib import Path


# Saved dashboard (always runs). Choose one optional detail table or None.
RESULTS_CSV = Path("results/qft_factorial_results.csv")
DETAIL_TABLE = None  # "false_alarm", "primary", "factors", or "leakage_watch"

# Posterior backend shared by model checks, the small demo, and the sweep.
BACKEND = "auto"  # "auto", "numpy", or "cupy"
GPU_PRECISION = "complex64"  # "complex64" or "complex128"

# Optional model checks and small repeated-secret example.
RUN_MODEL_CHECKS = False
RUN_DEMO = False
DEMO_RUNS = 8  # Example trials only. Use the saved sweep for rates.

# Optional Bob-side drift/recalibration demo (500 attempts by default).
RUN_RECALIBRATION_DEMO = False
RECALIBRATION_TRIALS = 512  # Evaluation paths per condition.
RECALIBRATION_CALIBRATION_TRIALS = 2_048  # Stationary paths for cutoffs.
RECALIBRATION_SEED = 20261122
RECALIBRATION_ATTEMPT_CAP = 500  # Must be a positive multiple of 5.

# Optional full 1,210-setting reproduction. Resume also requires matching
# BACKEND and GPU_PRECISION. The saved checkpoint is CuPy/complex64.
RUN_FULL_QFT_SWEEP = False
RUNS_PER_SETTING = 100  # Evaluation trials per grid setting.
PHASE_SAMPLES = 16  # Bob-phase samples in Eve's posterior likelihood.
CALIBRATION_RUNS = 100  # Noise-only paths per block/noise group.
BATCH_SIZE = 128  # Compute batching only, not checkpoint identity.
RESUME_SWEEP = True
CHECKPOINT_PATH = Path("results/qft_factorial_checkpoint.json")
SWEEP_RESULTS_CSV = Path("results/qft_factorial_reproduction.csv")

<a id="qft-dashboard"></a>

## Read the saved results

The saved run covers 2- and 4-qubit blocks, 11 fixed-noise profiles, and 100 trials per setting. Bob checks at 20, 50, and 100 retained bits, so unresolved sessions stop after about 25 attempts.

The left panel shows false alarms by noise profile. The pooled rate was 160/2,200 (7.27%) against a 5% target. The right panel shows alarms before recovery versus `K × f`, where `K` is Eve count and `f` is interception probability per Eve per qubit per leg. Zero attacked trials reached a 99%-confidence full-key declaration.

In the [settings cell](#qft-settings), set `DETAIL_TABLE` to `"false_alarm"`, `"primary"`, `"factors"`, or `"leakage_watch"`. See the [run notes](results/README.md) for fields and limitations.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from quantum_study.qft_results import build_qft_summary, plot_qft_summary

qft_factorial_results = pd.read_csv(RESULTS_CSV)
qft_dashboard = build_qft_summary(
    qft_factorial_results, expected_settings=1_210, target_false_alarm=0.05,
)
display(qft_dashboard["audit"])

qft_dashboard_figure = plot_qft_summary(
    qft_dashboard,
    block_sizes=sorted(qft_factorial_results["block_qubits"].unique()),
    noise_order=qft_factorial_results["noise_profile"].unique(),
)
plt.show()

if DETAIL_TABLE is not None:
    display(qft_dashboard[DETAIL_TABLE].round(4))

<a id="qft-methods"></a>

## Follow one exchange

1. **Send.** Bob prepares fresh private phases. Each leg applies Pauli noise.
2. **Encode and return.** Alice repeats the four-bit secret across blocks with equally many randomized hidden checks. Bob removes his phases and applies the inverse QFT.
3. **Infer.** Sequential Eves measure either leg and pool records. Public check disclosures and phase marginalization update a posterior over 16 secrets.
4. **Stop.** Bob accumulates joint check-pattern surprisal under the attack-free model. A trial ends at Bob's alarm, a 99%-confidence full-key declaration, or the check budget. Bob wins same-block ties.

The sweep calibrates each fixed-noise profile separately and reports full-key confidence. Measuring partial leakage requires per-bit posterior trajectories.

The exchange follows Papadopoulos and Linvill. The proposal supplies the inference and alarm objectives. See the [references](#references) and the implementations for [protocol physics](quantum_study/qft_protocol.py), [posterior and backends](quantum_study/qft_inference.py), [trial workflows](quantum_study/qft_trials.py), [sweep and checkpoints](quantum_study/qft_sweep.py), and [model checks](quantum_study/qft_validation.py).

<a id="qft-demo"></a>

## Run a small validation and demo

In the [settings cell](#qft-settings), set `RUN_MODEL_CHECKS = True` for noiseless decoding and likelihood checks. Set `RUN_DEMO = True` for `DEMO_RUNS` example trials under the Erven profile. Use the saved sweep for rate estimates.

`BACKEND = "auto"` uses CUDA when available and NumPy otherwise. Choose `"numpy"` or `"cupy"` to require one backend. See the [optional CUDA setup](README.md#optional-cuda).

In [ ]:
if RUN_MODEL_CHECKS or RUN_DEMO:
    from quantum_study.qft_inference import configure_qft_backend
    from quantum_study.qft_trials import run_qft_demo
    from quantum_study.qft_validation import validate_qft_model

    backend = configure_qft_backend(BACKEND, gpu_precision=GPU_PRECISION)
    print(f"Posterior backend: {backend.name} ({backend.precision})")
    if RUN_MODEL_CHECKS:
        validation = validate_qft_model(backend=backend, verbose=True)
    if RUN_DEMO:
        qft_recovery_results, demo_summary, qft_alarm_rule = run_qft_demo(
            evaluation_runs=DEMO_RUNS, backend=backend,
            return_diagnostics=True,
        )
        display(qft_recovery_results)
        display(demo_summary)

<a id="qft-recalibration"></a>

## Test channel drift and recalibration

Set `RUN_RECALIBRATION_DEMO = True` in the [settings cell](#qft-settings) for 500 attempts with four two-qubit blocks and four fresh checks per attempt. The same cell exposes the trial count, calibration count, seed, and attempt cap.

- **Channel.** Stable paths use `(q_Z, q_X) = (0.002, 0.038)`. Each drifting trial starts there and scales both rates together. Its seeded multiplier stays between 0.5× and 2×. Each attempt retains 98% of its last deviation from 1 and adds jitter, giving a nominal long-run multiplier SD of 0.30 before bounding. The drift-only and drift-plus-interception cases share these noise paths.
- **Interception.** Each Eve intercepts with 2.5% probability per qubit per leg. The two Eves redraw independently with Z probabilities of 0.1 and 0.9.
- **Alarm policies.** Both accumulate score deviations and test the same held-out checks every five attempts. Fixed keeps its expected window score. Frequent moves it halfway toward the observed total after testing the window. Each policy sets its own cutoff using the same 2,048 stationary noise-only calibration paths, targeting a 5% chance of any alarm across 100 looks.

Bob halts at the first alarm. A noise-only alarm is a false alarm. On Frequent rows, `fixed_only` counts paths where Fixed alarmed and Frequent reached the cap without an alarm. This demo averages over Eve's outcomes to evaluate Bob's alarms. [Implementation](quantum_study/qft_recalibration.py)

In [ ]:
if RUN_RECALIBRATION_DEMO:
    from matplotlib.ticker import PercentFormatter
    from quantum_study.qft_recalibration import run_qft_recalibration_demo

    recalibration_summary, recalibration_curves = run_qft_recalibration_demo(
        trials=RECALIBRATION_TRIALS, seed=RECALIBRATION_SEED,
        calibration_trials=RECALIBRATION_CALIBRATION_TRIALS,
        attempt_cap=RECALIBRATION_ATTEMPT_CAP,
    )
    display(recalibration_summary.assign(
        rate=recalibration_summary["rate"].map("{:.1%}".format),
    ))

    drifting_conditions = ("Drifting noise", "Drift and interception")
    curve_end = int(recalibration_curves["attempt"].max())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True, layout="constrained")
    for ax, condition in zip(axes, drifting_conditions):
        condition_curves = recalibration_curves[
            recalibration_curves["condition"].eq(condition)
        ]
        for policy, curve in condition_curves.groupby("policy", sort=False):
            ax.plot(
                curve["attempt"], curve["fraction_alarmed"],
                marker="o", markersize=3, label=policy,
            )
        ax.set(
            title=condition, xlabel="Attempt",
            xlim=(0, curve_end), ylim=(0, 1),
        )
        ax.grid(alpha=0.25)
    axes[0].set_ylabel("Cumulative fraction alarmed")
    axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[-1].legend(title="Baseline policy")
    fig.suptitle("Bob-side alarms under time-correlated channel noise")
    plt.show()


<a id="qft-batch"></a>

## Reproduce the full sweep

The 1,210-setting grid uses one common interception rate for all Eves. The saved RTX 5090 run used 100 trials per setting and took about 131 minutes. NumPy and CuPy use the same posterior calculation.

Set `RUN_FULL_QFT_SWEEP = True` in the [settings cell](#qft-settings) to reproduce it. Match settings to resume a checkpoint, or use new paths for a fresh run. After it finishes, point `RESULTS_CSV` to the reproduction CSV and rerun the dashboard to review it.

In [ ]:
if RUN_FULL_QFT_SWEEP:
    from quantum_study.qft_inference import configure_qft_backend
    from quantum_study.qft_sweep import run_qft_factorial_sweep

    backend = configure_qft_backend(BACKEND, gpu_precision=GPU_PRECISION)
    reproduced_results = run_qft_factorial_sweep(
        runs_per_setting=RUNS_PER_SETTING,
        checkpoint_path=CHECKPOINT_PATH,
        results_path=SWEEP_RESULTS_CSV,
        resume=RESUME_SWEEP,
        backend=backend,
        phase_samples=PHASE_SAMPLES,
        calibration_runs=CALIBRATION_RUNS,
        batch_size=BATCH_SIZE,
    )
    print(f"Saved {len(reproduced_results):,} settings to {SWEEP_RESULTS_CSV}")

<a id="references"></a>

## Methods and references

- **Papadopoulos and Linvill (2025), _Increasing Interference Detection in Quantum Cryptography Using the Quantum Fourier Transform_.** The scrambler, phase encoding, and inverse-QFT decoding correspond to equations (1)–(3) in §4.1 and Fig. 2. Interference propagation follows §4.2 and §§5.1–5.3, especially equations (7)–(11). The repeated-copy motivation and hidden checks follow §§3.2 and 6.1. [arXiv:2404.12507](https://arxiv.org/abs/2404.12507)
- **Dehmani et al. (2012), _Quantum Key Distribution with Several Intercept-Resend Attacks via a Depolarizing Channel_.** Motivates studying multiple sequential Eves. [arXiv:1302.7206](https://arxiv.org/abs/1302.7206)
- **Erven et al. (2009), _Entangled Quantum Key Distribution with a Biased Basis Choice_.** Supplies the reference noise-profile values used in the demo and sweep. [arXiv:0901.0960](https://arxiv.org/abs/0901.0960)
- **Napieraj and Chuang (2026), _Detecting Patient Attacks on a Repeated Secret_, §2.** The study proposal specifies the block sizes, channel and attack factors, joint check-pattern likelihoods, the 99% recovery outcome, and a 100-attempt cap. Phase sampling and the same-block timing rule are implementation choices. The saved run covers 25 attempts.

[Back to top](#top)